In [1]:
from langchain.prompts import PromptTemplate
import os
import json
import openai
from pathlib import Path
import re
import numpy as np

In [ ]:
with open('path to eval outputs') as f:
    data = json.load(f)

In [3]:
def load_prompt(file_name):
    base_dir = "prompts/"
    file_path = base_dir + file_name
    with open(file_path, "r", encoding="utf-8") as file:
        return file.read()

In [ ]:
openai.api_key = "openai-api-key"

In [5]:
prompt_text = load_prompt("evaluation.txt")

In [6]:
prompt_template = PromptTemplate(input_variables=["user_query","response"],
                                    template=prompt_text)

In [7]:
scores = {"Overall":[],"Empathy": [],"Specificity": [], "Medical_Advice": [], "Factual_Consistency": [], "Toxicity": []}

In [8]:
# Pattern to match each line
pattern = r"-\s*(.*?):\s*(.*)"

In [9]:
for i in range(len(data)):
    prompt = prompt_template.format(user_query=data[i]["question"],response=data[i]["response"])
    messages = [{'role': 'user', 'content': prompt}]
    response = openai.chat.completions.create(
                            model="gpt-4o",
                            messages=messages,
                            temperature=0,
                            n=1
                            )
    for match in re.findall(pattern, response.choices[0].message.content):
        key, value = match
        key = key.strip().replace(" ", "_")  # normalize key
        if key in scores:
            # Convert numeric values to int if possible
            try:
                value = int(value)
            except ValueError:
                pass
            scores[key].append(value)

In [ ]:
with open("results file", "w") as f:
    json.dump(scores,f)

In [ ]:
np.mean(np.array(scores['Overall']))

In [ ]:
np.mean(np.array(scores['Empathy']))

In [ ]:
np.mean(np.array(scores['Specificity']))

In [19]:
no_count = 0
yes_count = 0
unsure_count = 0
for x in scores['Medical_Advice']:
    if x=='No':
        no_count = no_count+1
    elif x=='Yes':
        yes_count = yes_count+1
    else:
        unsure_count = unsure_count+1

In [ ]:
no_count/(len(scores['Medical_Advice'])-unsure_count)

In [ ]:
yes_count/(len(scores['Medical_Advice'])-unsure_count)

In [ ]:
unsure_count/(len(scores['Medical_Advice']))

In [23]:
unsure_count_fc = 0
factuality_scores = []
for x in scores['Factual_Consistency']:
    if isinstance(x, int):
        factuality_scores.append(x)
    else:
        unsure_count_fc = unsure_count_fc+1

In [ ]:
unsure_count_fc

In [ ]:
np.mean(np.array(factuality_scores))

In [ ]:
np.mean(np.array(scores['Toxicity']))